# Reproduce: `norm_ab_v2`

**Axis:** normalization A/B unweighted

**Pins:** dataset=v2 seed=42 window=60.0s epochs=300 edge_weight=False scorer=stochastic

This notebook re-executes the same CLI as the original run into `nb_repro/` under this run dir, then compares frozen metrics to `reproduce_config.json` expected values.

```bash
cd .
.venv/bin/python -m abrg.validate_reproduce --run-dir abrg/output/norm_ab_v2
# or: jupyter nbconvert --to notebook --execute abrg/output/norm_ab_v2/reproduce.ipynb
```


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

CWD = Path.cwd().resolve()
REPO_ROOT = CWD if (CWD / "abrg").is_dir() else CWD.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from abrg.reproduce_kinds import (
    cli_argv_from_config,
    compare_kind_metrics,
    extract_metrics_from_result,
    result_json_name,
)

RUN_DIR = (REPO_ROOT / 'abrg/output/norm_ab_v2').resolve()
CFG = json.loads((RUN_DIR / "reproduce_config.json").read_text(encoding="utf-8"))
KIND = CFG.get("kind", "ratio")
PROFILE = CFG.get("profile", "")
RESULT_NAME = result_json_name(KIND)
EXPECTED_JSON = RUN_DIR / RESULT_NAME
print("RUN_DIR:", RUN_DIR)
print("kind:", KIND)
print("axis:", CFG.get("axis"))


In [ ]:
REPRO_DIR = RUN_DIR / "nb_repro"
if REPRO_DIR.exists():
    import shutil
    shutil.rmtree(REPRO_DIR)
REPRO_DIR.mkdir(parents=True)

argv = [sys.executable, *cli_argv_from_config(CFG, REPRO_DIR)]
print("Running:", " ".join(argv))
proc = subprocess.run(argv, cwd=REPO_ROOT)
print("exit_code:", proc.returncode)
assert proc.returncode == 0, "reproduce CLI failed"


In [ ]:
actual_data = json.loads((REPRO_DIR / RESULT_NAME).read_text(encoding="utf-8"))
actual = extract_metrics_from_result(actual_data, kind=KIND, profile=PROFILE)
expected = CFG.get("expected")
if expected is None:
    expected = extract_metrics_from_result(
        json.loads(EXPECTED_JSON.read_text(encoding="utf-8")),
        kind=KIND,
        profile=PROFILE,
    )
report = compare_kind_metrics(expected, actual, kind=KIND, config=CFG)
(RUN_DIR / "nb_validate_report.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print(json.dumps(report, indent=2))
assert report["ok"], "reproduce metrics outside tolerance — see nb_validate_report.json"
print("REPRODUCE OK")
